# Point 3 — shared 54-landmark experiment

**Run this GPU notebook once for H1 (per layer), H2 (per view), and H3 (per anchor).**
The three companion notebooks read its outputs without rerunning models.

The faster design uses **18 calibration + 36 confirmation landmarks**, each with **up to 50 eligible distinct photographs**.
Seven calibration conditions share their masks, baseline, ground truth and loaded VGGT model.
Freeze choices on calibration, then run only chosen conditions on confirmation. This is at most
**306 scene/condition jobs plus reference checks**, compared with 702 independent jobs.
The 79 final holdout landmarks remain untouched. This is preliminary evidence for point 3,
not a completed benchmark or a guarantee of finishing within an hour.

1. Keep this notebook and its source ZIP in `MyDrive/Point3_Shared54`.
2. Select a GPU runtime. **50-view memory requirements must pass the pilot**; a T4 has not been verified.
3. In Colab Secrets, create `HF_TOKEN`, enable notebook access, and use a token with approved SAM 3 access.
4. Run cells in order. The pilot must pass before the main run. Resume with the same settings/output path.

Forth Bridge uses the user-authorized manual anchor **bridge**. Washington Monument uses the
authorized **obelisk** correction after its supplied **altar** anchor produced 50 empty masks.
The original 18/36 split is preserved; obelisk uses H3's global-scalar fallback.
Every selected-view SAM output is checked;
saved overlays require visual review before architectural conclusions. Rare anchor types, including
dome and minaret, use the calibration-selected global scalar fallback.


In [ ]:
import hashlib, json, os, subprocess, sys, time, zipfile
from pathlib import Path
_previous_source = globals().get('SOURCE')
from google.colab import drive
drive.mount('/content/drive')

SHARED = Path('/content/drive/MyDrive/Point3_Shared54')
SOURCE_SHA256 = '83796943ffafa29f96a5fee1f4e58887a8b66b3da04515ebbd1375769ac4b446'
BUNDLE = SHARED / 'point3_shared54_source_83796943ffaf.zip'
CALIBRATION_SIZE = 18  # 18 calibration + 36 confirmation; 54 sweeps are optional.
RUN_LABEL = 'v1'       # Keep unchanged to resume; use a new label for a new study.
SESSION_HOURS = 8      # Session limit, NOT a predicted completion time.
DEVICES = '0'          # Use '0,1' only when two matching GPUs actually exist.
OUTPUT = SHARED / 'results' / f'shared54_{SOURCE_SHA256[:12]}_cal{CALIBRATION_SIZE}_{RUN_LABEL}'
OUTPUT.mkdir(parents=True, exist_ok=True)
SOURCE = Path('/content') / ('point3_shared54_source_' + SOURCE_SHA256[:12])
if not BUNDLE.exists():
    raise FileNotFoundError(f'Keep the source ZIP in {SHARED} alongside this notebook: {BUNDLE.name}')
if hashlib.sha256(BUNDLE.read_bytes()).hexdigest() != SOURCE_SHA256:
    raise ValueError('Source bundle checksum mismatch')
with zipfile.ZipFile(BUNDLE) as archive:
    if archive.testzip() is not None: raise ValueError('Damaged source bundle')
    for entry in archive.infolist():
        dest = (SOURCE / entry.filename).resolve()
        if not dest.is_relative_to(SOURCE.resolve()): raise ValueError('Unsafe bundle path')
        payload = archive.read(entry)
        if dest.exists() and dest.read_bytes() != payload:
            raise ValueError(f'Local source changed: {dest}; use a separate version directory')
        dest.parent.mkdir(parents=True, exist_ok=True)
        if not dest.exists(): dest.write_bytes(payload)
STUDY = SOURCE / 'experiments/point3/shared54'
if _previous_source is not None and Path(_previous_source).resolve() != SOURCE.resolve():
    for _name, _module in list(sys.modules.items()):
        _file = getattr(_module, '__file__', None)
        if _file and Path(_file).resolve().is_relative_to(Path(_previous_source).resolve()):
            sys.modules.pop(_name, None)
sys.path[:0] = [str(STUDY), str(STUDY.parent), str(SOURCE)]
WORK = Path('/content/adaptiveattend_point3')
EVAL_PY = WORK / 'envs/vggt/bin/python'
SAM_PY = WORK / 'envs/sam3/bin/python'
SCRATCH = Path('/content/point3_shared54_work')
DEADLINE = time.monotonic() + SESSION_HOURS*3600
print('Verified source:', SOURCE_SHA256[:12])
print('Persistent results:', OUTPUT)
print('54 landmarks; up to 50 eligible distinct photographs per landmark; actual counts recorded.')


In [ ]:
from protocol import make_plan, job_budget
PLAN = make_plan(CALIBRATION_SIZE)
print(json.dumps(job_budget(PLAN), indent=2))
print('Anchor types with sufficient calibration support:', PLAN['supported_anchor_types'])
print('Global-scalar fallback types:', PLAN['fallback_anchor_types'])
print('Manual override:', PLAN['anchor_overrides'])
print('Calibration IDs:', [r['scene'] for r in PLAN['rows'] if r['split']=='calibration'])


## Prepare environments

Reuses compatible installed environments; otherwise installs isolated, pinned VGGT and SAM environments.
This can take several minutes. Credentials are read from Secrets only when launching a run.


In [ ]:
from hypothesis_study import bounded_run, BudgetExceeded, write_json

def checked(command, label):
    print(label, flush=True)
    return bounded_run(command, OUTPUT / 'setup.log', SOURCE, os.environ.copy(), DEADLINE)

def pinned_repository(url, directory, revision):
    directory.parent.mkdir(parents=True, exist_ok=True)
    if not directory.exists():
        checked(['git', 'clone', '--quiet', url, directory], f'Cloning {directory.name}')
    # Read-only inspection; never overwrite a checkout with local edits.
    status = subprocess.check_output(['git', '-C', str(directory), 'status', '--porcelain'], text=True, timeout=10)
    if status.strip():
        raise RuntimeError(f'Local edits found in {directory}; resolve them before changing its revision')
    checked(['git', '-C', directory, 'fetch', '--quiet', 'origin', revision], f'Pinning {directory.name}')
    checked(['git', '-C', directory, 'checkout', '--quiet', revision], f'Checking {directory.name} revision')

eval_probe = ("import torch,numpy,safetensors,sys,importlib;"
              "assert torch.__version__.startswith('2.3.1'),torch.__version__;"
              "assert numpy.__version__=='2.2.6',numpy.__version__;"
              "assert torch.cuda.is_available(),'Select a GPU runtime';"
              "sys.path[:0]=['intervention','.'];importlib.import_module('models.vggt.run_pass')")
sam_probe = ("import torch,decord,pycocotools;from sam3.model_builder import build_sam3_image_model;"
             "from sam3.model.sam3_image_processor import Sam3Processor;"
             "assert torch.__version__.startswith('2.7.0'),torch.__version__;"
             "assert torch.cuda.is_available(),'SAM requires a GPU'")

def ready(python, probe, repo, revision):
    if not python.exists() or not (repo / '.git').exists():
        return False
    head = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', 'HEAD'], text=True, timeout=10).strip()
    dirty = subprocess.check_output(['git', '-C', str(repo), 'status', '--porcelain'], text=True, timeout=10).strip()
    if head != revision or dirty:
        return False
    try:
        checked([python, '-c', probe], f'Checking existing {python.parent.parent.name} environment')
        return True
    except BudgetExceeded:
        raise
    except RuntimeError:
        return False

vggt_revision = '97bbde571faddde3ace3cfa7724a20448026c4c8'
sam_revision = '11dec2936de97f2857c1f76b66d982d5a001155d'
vggt_dir, sam_dir = WORK / 'third_party/vggt', WORK / 'third_party/sam3'
try:
    eval_ready = ready(EVAL_PY, eval_probe, vggt_dir, vggt_revision)
    sam_ready = ready(SAM_PY, sam_probe, sam_dir, sam_revision)
    if not (eval_ready and sam_ready):
        checked([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], 'Preparing isolated environments')
        uv = [sys.executable, '-m', 'uv']
        if not eval_ready:
            if not EVAL_PY.exists():
                checked(uv + ['venv', '--python', '3.12', EVAL_PY.parent.parent], 'Creating VGGT environment')
            pip = uv + ['pip', 'install', '--python', EVAL_PY]
            checked(pip + ['torch==2.3.1', 'torchvision==0.18.1', '--index-url', 'https://download.pytorch.org/whl/cu121'], 'Installing VGGT GPU libraries')
            checked(pip + ['numpy==2.2.6', 'safetensors', '-r', SOURCE / 'requirements.txt'], 'Installing evaluation dependencies')
            pinned_repository('https://github.com/facebookresearch/vggt.git', vggt_dir, vggt_revision)
            checked(pip + ['--no-deps', '-e', vggt_dir], 'Installing VGGT')
        if not sam_ready:
            if not SAM_PY.exists():
                checked(uv + ['venv', '--python', '3.12', SAM_PY.parent.parent], 'Creating SAM 3 environment')
            pip = uv + ['pip', 'install', '--python', SAM_PY]
            checked(pip + ['setuptools<81', 'wheel'], 'Installing SAM build dependencies')
            checked(pip + ['torch==2.7.0', 'torchvision==0.22.0', '--index-url', 'https://download.pytorch.org/whl/cu126'], 'Installing SAM GPU libraries')
            pinned_repository('https://github.com/facebookresearch/sam3.git', sam_dir, sam_revision)
            checked(pip + ['-e', sam_dir, 'einops', 'psutil', 'numpy<2', 'pillow', 'decord==0.6.0', 'pycocotools'], 'Installing SAM 3 dependencies')
    checked([EVAL_PY, '-c', eval_probe], 'Validating VGGT imports and GPU')
    checked([SAM_PY, '-c', sam_probe], 'Validating SAM imports and GPU')
    # Preserve actual package versions and include them in checkpoint identity.
    for name, python in [('evaluation', EVAL_PY), ('sam', SAM_PY)]:
        versions_path = OUTPUT / (name+'_packages.json')
        script = ("import importlib.metadata,json,pathlib; "
                  f"pathlib.Path({str(versions_path)!r}).write_text(json.dumps(sorted((d.metadata['Name'],d.version) for d in importlib.metadata.distributions())))")
        bounded_run([python, '-c', script], OUTPUT / 'setup.log', SOURCE, os.environ.copy(), DEADLINE)
    print(f'Environments ready. {(DEADLINE-time.monotonic())/60:.1f} minutes remain in this session.')
except Exception as exc:
    write_json(OUTPUT / 'status.json', {'status': 'setup_incomplete', 'reason': type(exc).__name__, 'num_views': 50})
    raise


In [ ]:
def run_phase(phase):
    from google.colab import userdata
    from hypothesis_study import bounded_run
    remaining = (DEADLINE-time.monotonic())/3600
    if remaining <= 0:
        raise RuntimeError('Session budget expired. Rerun the settings cell to start a new session and resume.')
    environment = os.environ.copy()
    try:
        try:
            environment['HF_TOKEN'] = userdata.get('HF_TOKEN')
        except Exception:
            raise RuntimeError('Add HF_TOKEN in Colab Secrets and enable notebook access; SAM 3 access must be approved.') from None
        environment['HF_HUB_DISABLE_IMPLICIT_TOKEN'] = '0'
        command = [EVAL_PY, '-u', STUDY/'run.py', '--output', OUTPUT, '--work', SCRATCH,
            '--sam-python', SAM_PY, '--phase', phase, '--hours', str(remaining),
            '--calibration-size', str(CALIBRATION_SIZE), '--devices', DEVICES]
        bounded_run(command, OUTPUT/'notebook.log', SOURCE, environment, DEADLINE, echo=True)
    finally:
        environment.pop('HF_TOKEN', None)


## Pilot: memory, numerical equivalence and timing

Runs all seven calibration conditions on one landmark. A fresh reference run is compared against
the cached run with a reused model. Predictions must match exactly and metrics within the declared
tolerance. If memory, masks or equivalence fail, the bulk run stays blocked. No automatic reduction
in image count for memory occurs. Smaller source scenes retain all eligible views up to 50. Inspect logs and change the runtime if needed.


In [ ]:
PILOT_PASSED = False
run_phase('pilot')
checks = [json.loads(p.read_text()) for p in OUTPUT.glob('scenes/*/reference_check.json')]
if not any(c.get('status') == 'passed' for c in checks):
    raise RuntimeError('No successful reference check found. Inspect scene logs.')
PILOT_PASSED = True
for path in OUTPUT.glob('scenes/*/scene_timing.json'):
    print(path.parent.name, json.loads(path.read_text()))
print('Pilot passed. These measured scene times are not a dataset-wide runtime guarantee.')


## Complete or resume the shared study

Runs remaining calibration jobs, freezes scalar/layer/view/anchor choices, and then evaluates
the 36 confirmation landmarks. Validated checkpoints are reused. An incomplete calibration never
unlocks confirmation. A deadline or failed scene is recorded as partial, not successful completion.
Results and SAM masks persist in Drive. Local scene inputs are released after all conditions finish.


In [ ]:
if not globals().get('PILOT_PASSED', False):
    raise RuntimeError('Run and pass the pilot cell first.')
run_phase('all')


## Inspect results and masks

Open the H1/H2/H3 companion report notebooks after completion, or to inspect partial progress.
Negative structured-minus-scalar normalized Chamfer distance favors the structured rule.
Look at failures and no-intervention comparisons as well. Numerical mask checks do not establish
that SAM segmented the intended architectural object; review overlays, particularly Forth Bridge.


In [ ]:
from IPython.display import display, Image, Markdown
summary = OUTPUT/'summary.json'
if summary.exists(): print(json.dumps(json.loads(summary.read_text()), indent=2))
else: print('No report yet; inspect notebook.log and scene failure logs.')
bridge = hashlib.sha256('Forth_Bridge'.encode()).hexdigest()[:16]
preview = OUTPUT/'scenes'/bridge/'mask_preview.jpg'
if preview.exists():
    display(Markdown('### Forth Bridge — manual “bridge” anchor'))
    display(Image(filename=str(preview)))
    print(json.loads((preview.parent/'mask_validation.json').read_text()))
else: print('Forth Bridge masks have not been generated yet.')
print('Reports:', OUTPUT/'reports')
